In [1]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

In [2]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

In [7]:
# 定义数据加载函数，返回的是一个PyTorch数据迭代器
# data_arrays是数据元组:(features, labels)
def load_array(data_arrays, batch_size, is_train=True):
    # 返回的是将每个样本的特征和标签按行拼接在一起的TensorDataset对象
    # dataset[0]返回的是第一个样本的特征和标签，dataset[0][0]返回的是第一个样本的特征，dataset[0][1]返回的是第一个样本的标签
    dataset = data.TensorDataset(*data_arrays)  

    # DataLoader接收打包好的dataset，自动完成切分、打乱和多进程加载(可选)等操作
    # 返回一个可迭代对象，每次for循环都会"吐"出一个batch_size大小的样本，is_train=True表示训练集需要打乱顺序
    return data.DataLoader(dataset, batch_size, shuffle=is_train)  

batch_size = 10
data_iter = load_array((features, labels), batch_size)

# data_iter只是一个可迭代对象，需要使用iter()函数将其转换为迭代器，才能使用next()函数获取下一个batch_size大小的样本
next(iter(data_iter))  # 返回一个batch_size大小的样本

[tensor([[ 0.2673,  0.7028],
         [-1.5208, -0.5000],
         [-0.4541,  0.0387],
         [ 0.4157,  0.0030],
         [-0.8029, -0.9704],
         [ 1.5613,  0.4821],
         [ 0.1478, -0.2339],
         [ 0.6974, -0.8025],
         [-1.3324, -0.8100],
         [ 0.8400,  0.5284]]),
 tensor([[2.3241],
         [2.8651],
         [3.1832],
         [5.0390],
         [5.9070],
         [5.6692],
         [5.2922],
         [8.3320],
         [4.2827],
         [4.0828]])]

In [ ]:
# nn是神经网络的缩写
from torch import nn

# Sequential是一个容器，里面用来放置一系列的神经网络层，按顺序依次执行
# Linear是一个线性层，输入特征数为2，输出特征数为1
net = nn.Sequential(nn.Linear(2, 1))  

In [12]:
# 初始化模型参数
# 初始化w的值为均值为0，标准差为0.01的正态分布随机数
net[0].weight.data.normal_(0, 0.01)  
# 初始化b的值为0
net[0].bias.data.fill_(0)

tensor([0.])

In [13]:
# 计算均方误差，直接用PyTorch提供的MSELoss函数
loss = nn.MSELoss()

In [14]:
# 实例化SGD(小批量梯度随即下降)实例,传入神经网络的参数(w,b)和学习率lr
trainer = torch.optim.SGD(net.parameters(), lr=0.03)


In [15]:
# 模型训练
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:  # for循环会自动调用iter()函数
        l = loss(net(X), y)  # 计算损失
        trainer.zero_grad()  # 梯度清零
        l.backward()  # 反向传播计算梯度
        trainer.step()  # 更新参数
    l = loss(net(features), labels)  # 计算整个训练集的损失
    print(f'epoch {epoch + 1}, loss {l:f}')  # 打印每个epoch的损失

epoch 1, loss 0.000328
epoch 2, loss 0.000098
epoch 3, loss 0.000100
